# Performance Internals — 01: Query Plan Deep Dive

## What you will learn
Before optimizing a Spark job, you need to understand **what Spark is actually doing**.
The query plan is Spark's blueprint — it shows every operation from your DataFrame code
down to the physical execution on disk.

In this notebook you will:
1. Understand the 4 stages of Spark query planning
2. Read and interpret `explain()` output
3. Identify common performance anti-patterns in plans
4. Use the Spark UI to correlate plan nodes with metrics
5. Write queries that produce efficient plans

## The 4 stages of query planning
```
Your DataFrame code
       │
       ▼
  Unresolved Logical Plan   ← names not yet checked
       │  Analyzer
       ▼
  Resolved Logical Plan     ← names resolved against catalog
       │  Optimizer (Catalyst)
       ▼
  Optimized Logical Plan    ← predicates pushed down, constants folded
       │  Physical Planner
       ▼
  Physical Plan             ← actual execution strategy (HashJoin vs SortMergeJoin etc.)
       │  Code Generation
       ▼
  Executed Plan             ← JVM bytecode or Velox native operators
```


In [ ]:
import os, time, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('performance-internals')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")

## Step 1 — Your First explain()

`explain()` has several modes — each reveals a different layer:

| Mode | Shows |
|---|---|
| `simple` | Physical plan only (default) |
| `extended` | Logical + Physical plan |
| `codegen` | Generated JVM bytecode |
| `cost` | Plan with estimated row counts and sizes |
| `formatted` | Clean formatted physical plan (most readable) |

Let's start with a simple query and inspect all modes.


In [ ]:
# Create a dataset with enough complexity to generate an interesting plan
import random
random.seed(42)

products_data = [(i, f"Product_{i}", random.choice(["Electronics","Clothing","Food","Books"]),
                  round(random.uniform(5, 500), 2), random.randint(0, 1000))
                 for i in range(1, 201)]

orders_data = [(i, random.randint(1, 200), random.randint(1, 50),
                round(random.uniform(1, 10), 2), f"2023-{random.randint(1,12):02d}-{random.randint(1,28):02d}")
               for i in range(1, 1001)]

products = spark.createDataFrame(products_data,
    ["product_id","name","category","price","stock"])
orders = spark.createDataFrame(orders_data,
    ["order_id","product_id","quantity","discount","order_date"])

products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")

print(f"products: {products.count()} rows")
print(f"orders  : {orders.count()} rows")

In [ ]:
# A query that joins, filters, and aggregates
query = (
    orders.join(products, "product_id")
          .filter(F.col("price") > 50)
          .filter(F.col("stock") > 0)
          .groupBy("category")
          .agg(
              F.sum(F.col("quantity") * F.col("price") * (1 - F.col("discount"))).alias("total_revenue"),
              F.countDistinct("order_id").alias("order_count"),
              F.avg("price").alias("avg_price")
          )
          .orderBy(F.desc("total_revenue"))
)

print("SIMPLE (default):")
query.explain()

In [ ]:
print("FORMATTED (most readable for learning):")
query.explain(mode="formatted")

In [ ]:
print("COST (shows estimated rows and bytes — used by Catalyst optimizer):")
query.explain(mode="cost")

## Step 2 — Understanding Physical Plan Operators

These are the most common operators you will see:

| Operator | Meaning | Performance note |
|---|---|---|
| `FileScan` / `Scan` | Reading data from storage | Check `PushedFilters` |
| `Filter` | Row filtering | Should appear close to scan (pushed down) |
| `Project` | Column selection/transformation | Cheap |
| `HashAggregate` | Aggregation using hash map | 2 phases: partial + final |
| `Exchange` | Data shuffle between executors | **Expensive — minimize these** |
| `Sort` | Sorting | Often follows Exchange |
| `SortMergeJoin` | Join via sort + merge | Default for large tables |
| `BroadcastHashJoin` | Join by broadcasting small table | Fast — no shuffle |
| `BroadcastExchange` | Sending table to all executors | Triggered by broadcast join |
| `WholeStageCodegen` | JVM bytecode generation wrapper | Good — operators fused |


In [ ]:
# Count Exchange nodes — each = one shuffle = expensive network I/O
plan_str = query._jdf.queryExecution().executedPlan().toString()
exchange_count = plan_str.count("Exchange")
sort_count = plan_str.count("Sort ")
join_type = "BroadcastHashJoin" if "BroadcastHashJoin" in plan_str else "SortMergeJoin"

print(f"Query plan analysis:")
print(f"  Exchange (shuffle) nodes : {exchange_count}")
print(f"  Sort nodes               : {sort_count}")
print(f"  Join strategy            : {join_type}")
print()
print("Rule of thumb: every Exchange = network transfer of ALL data.")
print("Fewer exchanges = faster job.")

## Step 3 — Catalyst Optimizer: Predicate Pushdown

One of Catalyst's most powerful optimizations: **push filters as close to the data source
as possible**, so we read less data.

Watch what happens when filters are defined in different positions.


In [ ]:
# ANTI-PATTERN: filter AFTER join (Catalyst may still push it down, but let's verify)
late_filter = (
    orders.join(products, "product_id")
          .filter(F.col("category") == "Electronics")  # filter after join
)

# BEST PRACTICE: filter BEFORE join
early_filter = (
    orders.join(
        products.filter(F.col("category") == "Electronics"),  # filter before join
        "product_id"
    )
)

plan_late  = late_filter._jdf.queryExecution().executedPlan().toString()
plan_early = early_filter._jdf.queryExecution().executedPlan().toString()

print("Late filter plan exchange count  :", plan_late.count("Exchange"))
print("Early filter plan exchange count :", plan_early.count("Exchange"))
print()
print("Catalyst often pushes predicates down automatically (predicate pushdown).")
print("But explicit early filtering makes intent clear and helps with Parquet pushdown.")

In [ ]:
# Verify predicate pushdown with Parquet
import pathlib
pathlib.Path(f"{DATA_DIR}/products_parquet").mkdir(parents=True, exist_ok=True)
products.write.mode("overwrite").parquet(f"{DATA_DIR}/products_parquet")

# Reading from Parquet — check PushedFilters in the plan
pq_scan = spark.read.parquet(f"{DATA_DIR}/products_parquet") \
               .filter(F.col("price") > 100) \
               .filter(F.col("category") == "Electronics")

print("Parquet scan with filters:")
pq_scan.explain(mode="formatted")
print()
print("Look for 'PushedFilters' in the FileScan node.")
print("Pushed filters are evaluated by the Parquet reader — no Spark task needed.")

## Step 4 — Join Strategies: When Spark Chooses What

Spark has 5 join strategies. Which one is chosen depends on table sizes and config.

| Strategy | When used | Shuffle needed |
|---|---|---|
| `BroadcastHashJoin` | Small table < `spark.sql.autoBroadcastJoinThreshold` (10MB default) | NO |
| `ShuffledHashJoin` | Medium tables, enough memory | YES (one side) |
| `SortMergeJoin` | Large tables (default for big joins) | YES (both sides) |
| `BroadcastNestedLoopJoin` | Non-equi join, small table | NO |
| `CartesianProduct` | Cross join (dangerous!) | NO |

Let's control the join strategy explicitly.


In [ ]:
# Force BroadcastHashJoin by hinting
broadcast_join = orders.join(F.broadcast(products), "product_id")
smj_join = orders.hint("merge").join(products, "product_id")

def get_join_type(df):
    plan = df._jdf.queryExecution().executedPlan().toString()
    for jtype in ["BroadcastHashJoin","SortMergeJoin","ShuffledHashJoin"]:
        if jtype in plan:
            return jtype
    return "Unknown"

print(f"Broadcast hint  → {get_join_type(broadcast_join)}")
print(f"Merge hint      → {get_join_type(smj_join)}")
print()

# Time them
t0 = time.time(); broadcast_join.count(); t_broadcast = time.time() - t0
t0 = time.time(); smj_join.count();       t_smj = time.time() - t0

print(f"BroadcastHashJoin time : {t_broadcast:.3f}s")
print(f"SortMergeJoin time     : {t_smj:.3f}s")
print(f"Speedup                : {t_smj/t_broadcast:.1f}x  (bigger gap with larger data)")

## Step 5 — Adaptive Query Execution (AQE)

AQE is Spark's **runtime optimizer** — it re-optimizes the plan *during* execution
using actual statistics collected at shuffle boundaries.

Three main AQE features:
1. **Coalescing shuffle partitions** — merges small post-shuffle partitions
2. **Converting SortMergeJoin to BroadcastHashJoin** — if one side turns out small
3. **Skew join optimization** — splits oversized partitions

Let's observe AQE in action.


In [ ]:
# AQE is already enabled in spark-defaults.conf
# Let's verify and see the runtime statistics

print("AQE settings:")
print(f"  spark.sql.adaptive.enabled                    : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"  spark.sql.adaptive.coalescePartitions.enabled : {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")
print()

# Create a skewed dataset to see AQE skew handling
import random
random.seed(42)

# 80% of orders from product_id=1 — extreme skew
skewed_data = [(i, 1 if random.random() < 0.8 else random.randint(2,50),
                random.randint(1,10)) for i in range(1, 5001)]
skewed_orders = spark.createDataFrame(skewed_data, ["order_id","product_id","qty"])

result = skewed_orders.groupBy("product_id").agg(F.sum("qty").alias("total_qty"))

print("Query on skewed data — AQE will detect and handle the skew:")
result.explain(mode="formatted")

t0 = time.time()
result.show(5)
print(f"\nExecution time: {time.time()-t0:.3f}s")
print("\nWith AQE: Spark detected the skew and split the heavy partition.")

## Step 6 — Reading the Spark UI (Conceptual Guide)

While this notebook runs in JupyterLab, the **Spark UI at http://localhost:4040**
shows real-time execution details.

### What to look for in the SQL tab
- **Duration** of each query stage
- **Rows read vs rows output** — large reduction = filter working well
- **Shuffle read/write bytes** — should be minimized
- **Skew** — one task taking 10x longer than others

### What to look for in the Stages tab
- Tasks with very long duration (skew indicator)
- High GC time (memory pressure)
- Large shuffle spill (not enough memory for sort)

### The key metric: Task Duration Distribution
A healthy job has a **tight distribution** of task durations.
A skewed job has a **long tail** — one or few tasks taking most of the time.


In [ ]:
# Generate a job that produces interesting UI metrics
# Check http://localhost:4040 while this runs

print("Running a multi-stage job — check Spark UI at http://localhost:4040")
print()

# Stage 1: Read and filter
stage1 = orders.filter(F.col("quantity") > 3).cache()
count1 = stage1.count()
print(f"Stage 1 — filtered orders: {count1:,}")

# Stage 2: Join
stage2 = stage1.join(products, "product_id").cache()
count2 = stage2.count()
print(f"Stage 2 — joined rows: {count2:,}")

# Stage 3: Aggregate
stage3 = (stage2.groupBy("category", F.month(F.to_date("order_date")).alias("month"))
                .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue"))
                .orderBy("category", "month"))
stage3.show(20)

# Cleanup
stage1.unpersist()
stage2.unpersist()
print("\nDone. Review the completed jobs in Spark UI History at http://localhost:18080")

## Summary

### The Query Plan Reading Checklist
When you see a slow Spark job, open `explain(mode="formatted")` and check:

1. **Where are the Exchange nodes?** — Each = shuffle. Can any be eliminated?
2. **Are filters close to the scan?** — Look for `PushedFilters` in FileScan
3. **What join strategy is used?** — BroadcastHashJoin >> SortMergeJoin for small tables
4. **Is AQE doing anything?** — Look for `AdaptiveSparkPlan` at the top
5. **How many stages?** — Each shuffle boundary = new stage = potential bottleneck

### Quick wins
| Problem | Fix |
|---|---|
| SortMergeJoin on small table | Use `F.broadcast()` hint |
| Filters not pushed to source | Move filters before join / use Parquet |
| Too many partitions after shuffle | AQE handles it, or set `spark.sql.shuffle.partitions` |
| Data skew | AQE skew join, or manual salting |

**Next:** `02_join_strategies.ipynb` — deep dive into each join type with benchmarks.
